# RQ2 + RQ3 Standalone Notebook

This notebook contains the complete code for Analyst 2. It does **not** call external Python scripts.

- **RQ2:** ZIP-level K-Means community profiling
- **RQ3:** Response-time comparison across ZIP clusters

Required input file: `311_2025_manhattan_cleaned.csv`.
The notebook will look for the CSV in the current folder, parent folder, or `source/`.


In [ ]:
from pathlib import Path
import itertools
import json
import math
import re
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation, PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

TOPIC_NAMES = {
    0: "Sanitation & Property Conditions",
    1: "Residential Noise & Street Disruption",
    2: "Parking & Construction Violations",
    3: "Building Noise & Vendor Issues",
}
THEME_ORDER = list(TOPIC_NAMES.values())
CLUSTER_LETTERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
RANDOM_STATE = 617

def find_project_root_and_data():
    cwd = Path.cwd()
    candidates = [
        cwd / "311_2025_manhattan_cleaned.csv",
        cwd / "source" / "311_2025_manhattan_cleaned.csv",
        cwd.parent / "311_2025_manhattan_cleaned.csv",
        cwd.parent / "source" / "311_2025_manhattan_cleaned.csv",
    ]
    for path in candidates:
        if path.exists():
            if path.parent.name == "source":
                return path.parent.parent, path
            return path.parent, path
    raise FileNotFoundError(
        "Could not find 311_2025_manhattan_cleaned.csv. "
        "Place it in the notebook folder, repo root, or source/."
    )

PROJECT_ROOT, DATA_PATH = find_project_root_and_data()
OUT_DIR = PROJECT_ROOT / "outputs" / "analysis2"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Output directory:", OUT_DIR)


In [ ]:
def clean_descriptor(value):
    text = "" if pd.isna(value) else str(value).lower()
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def slug(value):
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def wrap_label(label, width=24):
    return "\n".join(textwrap.wrap(label, width=width, break_long_words=False))


def prepare_data(path):
    df = pd.read_csv(path, low_memory=False)
    df["incident_zip"] = df["incident_zip"].astype(str)
    df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
    df["closed_date"] = pd.to_datetime(df["closed_date"], errors="coerce")
    df["response_hours"] = pd.to_numeric(df["response_hours"], errors="coerce").clip(lower=0)
    df["log_response_hours"] = np.log1p(df["response_hours"])
    df["descriptor_clean"] = df["descriptor"].map(clean_descriptor)
    df["has_descriptor"] = df["descriptor_clean"].str.len() > 0
    return df


data = prepare_data(DATA_PATH)
print("Shape:", data.shape)
print("ZIP codes:", data["incident_zip"].nunique())
print("Date range:", data["created_date"].min(), "to", data["created_date"].max())
data.head()


## RQ2: ZIP-Level Community Profile Clustering

The RQ2 workflow is:

1. Reproduce the four LDA topic labels from complaint descriptors.
2. Aggregate topic shares to ZIP level.
3. Run K-Means with `k=4`.
4. Use PCA and topic-share heatmap to interpret clusters.


In [ ]:
def add_lda_topics(df, random_state=RANDOM_STATE):
    desc_mask = df["has_descriptor"]
    desc_text = df.loc[desc_mask, "descriptor_clean"]

    vectorizer = CountVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=100,
        ngram_range=(1, 2),
    )
    count_matrix = vectorizer.fit_transform(desc_text)

    lda = LatentDirichletAllocation(
        n_components=4,
        random_state=random_state,
        max_iter=50,
    )
    topic_distribution = lda.fit_transform(count_matrix)
    dominant_topic = topic_distribution.argmax(axis=1)

    df = df.copy()
    df["topic_id"] = np.nan
    df.loc[desc_mask, "topic_id"] = dominant_topic
    df["topic_name"] = df["topic_id"].map(TOPIC_NAMES)

    feature_names = vectorizer.get_feature_names_out()
    topic_rows = []
    for idx, topic in enumerate(lda.components_):
        top_idx = topic.argsort()[-12:][::-1]
        topic_rows.append({
            "topic_id": idx,
            "topic_name": TOPIC_NAMES[idx],
            "top_terms": ", ".join(feature_names[i] for i in top_idx),
        })
    return df, pd.DataFrame(topic_rows)


data, topic_terms = add_lda_topics(data)
topic_terms.to_csv(TABLE_DIR / "lda_topic_terms_reproduced.csv", index=False)
topic_terms


In [ ]:
def build_zip_features(df, top_n_complaints=20, min_zip_records=50):
    zip_counts = df["incident_zip"].value_counts()
    keep_zips = zip_counts[zip_counts >= min_zip_records].index
    df_model = df[df["incident_zip"].isin(keep_zips)].copy()

    top_complaints = df_model["complaint_type"].value_counts().head(top_n_complaints).index
    complaint_matrix = pd.crosstab(
        df_model["incident_zip"],
        df_model["complaint_type"],
        normalize="index",
    )
    complaint_matrix = complaint_matrix.reindex(columns=top_complaints, fill_value=0)
    complaint_matrix.columns = [f"complaint_share__{slug(col)}" for col in complaint_matrix.columns]

    topic_valid = df_model.dropna(subset=["topic_name"]).copy()
    topic_matrix = pd.crosstab(
        topic_valid["incident_zip"],
        topic_valid["topic_name"],
        normalize="index",
    )
    topic_matrix = topic_matrix.reindex(columns=THEME_ORDER, fill_value=0)
    topic_matrix.columns = [f"topic_share__{slug(col)}" for col in topic_matrix.columns]

    zip_basic = df_model.groupby("incident_zip").agg(
        total_complaints=("unique_key", "count"),
    )
    zip_basic["log_total_complaints"] = np.log1p(zip_basic["total_complaints"])

    zip_features = zip_basic.join(complaint_matrix, how="left").join(topic_matrix, how="left")
    zip_features = zip_features.fillna(0).reset_index()

    feature_columns = [col for col in zip_features.columns if col.startswith("topic_share__")]
    return zip_features, feature_columns


def kmeans_diagnostics(scaled, k_min=2, k_max=8, random_state=RANDOM_STATE):
    rows = []
    for k in range(k_min, min(k_max, len(scaled) - 1) + 1):
        model = KMeans(n_clusters=k, n_init=50, random_state=random_state)
        labels = model.fit_predict(scaled)
        rows.append({
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(scaled, labels),
        })
    return pd.DataFrame(rows)


def assign_cluster_labels(zip_features):
    topic_cols = [f"topic_share__{slug(name)}" for name in THEME_ORDER]
    profile = zip_features.groupby("kmeans_cluster")[topic_cols].mean()
    row_idx, col_idx = linear_sum_assignment(-profile.to_numpy())
    cluster_meta = []
    for row_pos, col_pos in zip(row_idx, col_idx):
        cluster_id = profile.index[row_pos]
        theme = THEME_ORDER[col_pos]
        cluster_meta.append((cluster_id, col_pos, theme))
    cluster_meta.sort(key=lambda item: (item[1], item[0]))
    return {
        cluster_id: f"Cluster {CLUSTER_LETTERS[theme_pos]}: {theme}"
        for cluster_id, theme_pos, theme in cluster_meta
    }


zip_features, feature_columns = build_zip_features(data)
scaled = StandardScaler().fit_transform(zip_features[feature_columns])
diagnostics = kmeans_diagnostics(scaled)

kmeans = KMeans(n_clusters=4, n_init=50, random_state=RANDOM_STATE)
zip_features["kmeans_cluster"] = kmeans.fit_predict(scaled)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_values = pca.fit_transform(scaled)
zip_features["pca1"] = pca_values[:, 0]
zip_features["pca2"] = pca_values[:, 1]
zip_features["cluster_label"] = zip_features["kmeans_cluster"].map(assign_cluster_labels(zip_features))

cluster_lookup = zip_features.set_index("incident_zip")["cluster_label"]
data["cluster_label"] = data["incident_zip"].map(cluster_lookup)
data = data.dropna(subset=["cluster_label"]).copy()

zip_features.to_csv(TABLE_DIR / "zip_cluster_assignments.csv", index=False)
diagnostics.to_csv(TABLE_DIR / "kmeans_diagnostics.csv", index=False)
data[["unique_key", "incident_zip", "complaint_type", "descriptor", "topic_name", "cluster_label"]].to_csv(
    TABLE_DIR / "rq2_row_topics.csv", index=False
)

print("ZIP codes clustered:", zip_features["incident_zip"].nunique())
print("PCA explained variance ratio:", pca.explained_variance_ratio_)
diagnostics


In [ ]:
def save_diagnostic_plot(diagnostics):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(diagnostics["k"], diagnostics["inertia"], marker="o", color="#2f6f73")
    axes[0].set_title("Elbow Method")
    axes[0].set_xlabel("Number of clusters (k)")
    axes[0].set_ylabel("Inertia")
    axes[0].grid(alpha=0.25)

    axes[1].plot(diagnostics["k"], diagnostics["silhouette"], marker="o", color="#8a5a44")
    axes[1].set_title("Silhouette Score")
    axes[1].set_xlabel("Number of clusters (k)")
    axes[1].set_ylabel("Silhouette")
    axes[1].grid(alpha=0.25)
    fig.suptitle("K-Means Diagnostics for ZIP-Level Issue Profiles", y=1.04, fontsize=14)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "kmeans_elbow_silhouette.png", dpi=220, bbox_inches="tight")
    plt.show()


def save_pca_plot(zip_features):
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.scatterplot(
        data=zip_features,
        x="pca1",
        y="pca2",
        hue="cluster_label",
        size="total_complaints",
        sizes=(70, 260),
        palette="Set2",
        ax=ax,
        edgecolor="white",
        linewidth=0.8,
    )
    for _, row in zip_features.iterrows():
        ax.text(row["pca1"] + 0.04, row["pca2"] + 0.04, row["incident_zip"], fontsize=8)
    ax.set_title("ZIP Clusters in PCA Space")
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "zip_pca_clusters.png", dpi=220, bbox_inches="tight")
    plt.show()


def save_theme_heatmap(zip_features):
    topic_cols = [f"topic_share__{slug(name)}" for name in THEME_ORDER]
    heatmap_data = zip_features.groupby("cluster_label")[topic_cols].mean()
    heatmap_data.columns = THEME_ORDER
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".1%",
        cmap="YlGnBu",
        linewidths=0.5,
        cbar_kws={"label": "Average ZIP topic share"},
        ax=ax,
    )
    ax.set_title("Cluster Alignment with Analyst 1 RQ1 Themes")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([wrap_label(label, 20) for label in heatmap_data.columns], rotation=0)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "cluster_theme_heatmap.png", dpi=220, bbox_inches="tight")
    plt.show()


save_diagnostic_plot(diagnostics)
save_pca_plot(zip_features)
save_theme_heatmap(zip_features)


## RQ3: Response-Time Differences Across Clusters

Response time is analyzed **after** clusters are assigned. It is not used to form the clusters.

Because `response_hours` is strongly right-skewed, the main test is Kruskal-Wallis. ANOVA on `log1p(response_hours)` is used as a robustness check, followed by pairwise Mann-Whitney U tests with Holm correction.


In [ ]:
def summarize_cluster_profiles(df, assignments):
    topic_cols = [f"topic_share__{slug(name)}" for name in THEME_ORDER]
    zip_topic_profile = assignments.groupby("cluster_label")[topic_cols].mean()
    zip_topic_profile.columns = THEME_ORDER

    profile_rows = []
    for cluster_label, rows in df.groupby("cluster_label"):
        top_complaints = rows["complaint_type"].value_counts(normalize=True).head(5)
        zips = rows["incident_zip"].drop_duplicates().sort_values()
        top_zips = rows["incident_zip"].value_counts().head(6)

        topic_share = rows["topic_name"].value_counts(normalize=True).reindex(THEME_ORDER).fillna(0)
        assigned_theme = cluster_label.split(": ", 1)[1] if ": " in cluster_label else topic_share.idxmax()
        strongest_theme = topic_share.idxmax()
        response = rows["response_hours"]

        profile_rows.append({
            "cluster_label": cluster_label,
            "zip_count": len(zips),
            "record_count": len(rows),
            "zips": ", ".join(zips.astype(str)),
            "top_zips_by_records": "; ".join(f"{idx} ({val})" for idx, val in top_zips.items()),
            "assigned_topic": assigned_theme,
            "assigned_topic_share": topic_share.get(assigned_theme, 0),
            "strongest_topic": strongest_theme,
            "strongest_topic_share": topic_share.max(),
            "topic_mix": "; ".join(f"{idx}: {val:.1%}" for idx, val in topic_share.items()),
            "top_complaints": "; ".join(f"{idx}: {val:.1%}" for idx, val in top_complaints.items()),
            "median_response_hours": response.median(),
            "mean_response_hours": response.mean(),
            "p90_response_hours": response.quantile(0.90),
        })
    return pd.DataFrame(profile_rows).sort_values("cluster_label")


def holm_adjust(p_values):
    indexed = sorted(enumerate(p_values), key=lambda item: item[1])
    adjusted = [math.nan] * len(p_values)
    running_max = 0.0
    m = len(p_values)
    for rank, (original_idx, p_value) in enumerate(indexed):
        corrected = min((m - rank) * p_value, 1.0)
        running_max = max(running_max, corrected)
        adjusted[original_idx] = running_max
    return adjusted


def response_tests(df):
    groups = [group["response_hours"].dropna().to_numpy() for _, group in df.groupby("cluster_label")]
    log_groups = [group["log_response_hours"].dropna().to_numpy() for _, group in df.groupby("cluster_label")]

    kw_stat, kw_p = stats.kruskal(*groups)
    anova_stat, anova_p = stats.f_oneway(*log_groups)
    omnibus = pd.DataFrame([
        {
            "test": "Kruskal-Wallis on response_hours",
            "statistic": kw_stat,
            "p_value": kw_p,
            "interpretation": "Nonparametric test for response-time differences across clusters",
        },
        {
            "test": "One-way ANOVA on log1p(response_hours)",
            "statistic": anova_stat,
            "p_value": anova_p,
            "interpretation": "Parametric robustness check after log transform",
        },
    ])

    summary = (
        df.groupby("cluster_label")["response_hours"]
        .agg(
            n="count",
            mean="mean",
            median="median",
            q1=lambda s: s.quantile(0.25),
            q3=lambda s: s.quantile(0.75),
            p90=lambda s: s.quantile(0.90),
        )
        .reset_index()
    )

    pair_rows = []
    labels = sorted(df["cluster_label"].unique())
    for left, right in itertools.combinations(labels, 2):
        left_values = df.loc[df["cluster_label"] == left, "response_hours"].dropna()
        right_values = df.loc[df["cluster_label"] == right, "response_hours"].dropna()
        stat, p_value = stats.mannwhitneyu(left_values, right_values, alternative="two-sided")
        pair_rows.append({
            "cluster_1": left,
            "cluster_2": right,
            "mann_whitney_u": stat,
            "p_value": p_value,
            "median_1": left_values.median(),
            "median_2": right_values.median(),
        })
    pairwise = pd.DataFrame(pair_rows)
    pairwise["p_value_holm"] = holm_adjust(pairwise["p_value"].tolist())
    pairwise["significant_0_05"] = pairwise["p_value_holm"] < 0.05
    return omnibus, pairwise, summary


cluster_profiles = summarize_cluster_profiles(data, zip_features)
omnibus_tests, pairwise_tests, response_summary = response_tests(data)

cluster_profiles.to_csv(TABLE_DIR / "cluster_profiles.csv", index=False)
omnibus_tests.to_csv(TABLE_DIR / "response_time_omnibus_tests.csv", index=False)
pairwise_tests.to_csv(TABLE_DIR / "response_time_pairwise_tests.csv", index=False)
response_summary.to_csv(TABLE_DIR / "response_time_summary_by_cluster.csv", index=False)

cluster_profiles


In [ ]:
def save_response_boxplot(df):
    order = sorted(df["cluster_label"].unique())
    plot_df = df.copy()
    plot_df["cluster_label_wrapped"] = plot_df["cluster_label"].map(lambda x: wrap_label(x, 24))
    order_wrapped = [wrap_label(label, 24) for label in order]

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(
        data=plot_df,
        x="cluster_label_wrapped",
        y="log_response_hours",
        hue="cluster_label_wrapped",
        order=order_wrapped,
        palette="Set2",
        showfliers=False,
        legend=False,
        ax=ax,
    )
    ax.set_title("Response Time Distribution by ZIP Cluster")
    ax.set_xlabel("")
    ax.set_ylabel("log1p(response hours)")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "response_time_boxplot.png", dpi=220, bbox_inches="tight")
    plt.show()


def save_profile_table_image(profile):
    display = pd.DataFrame({
        "Cluster / Theme": profile["cluster_label"].map(lambda x: wrap_label(x, 28)),
        "Size": profile.apply(lambda row: f"{int(row['zip_count'])} ZIPs\n{int(row['record_count']):,} records", axis=1),
        "Top Complaint Mix": profile["top_complaints"].map(lambda x: "\n".join(x.split("; ")[:3])),
        "Response Time": profile.apply(
            lambda row: f"Median: {row['median_response_hours']:.1f} hrs\nP90: {row['p90_response_hours']:.1f} hrs",
            axis=1,
        ),
    })

    fig, ax = plt.subplots(figsize=(8.8, 5.4))
    ax.axis("off")
    ax.text(0.5, 0.96, "Cluster Profile Table", ha="center", va="top", fontsize=15, weight="bold", transform=ax.transAxes)
    table = ax.table(
        cellText=display.values,
        colLabels=display.columns,
        cellLoc="left",
        colLoc="left",
        colWidths=[0.29, 0.14, 0.36, 0.18],
        bbox=[0.02, 0.04, 0.96, 0.82],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8.5)
    table.scale(1, 2.9)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#d4d4d4")
        cell.PAD = 0.12
        if row == 0:
            cell.set_facecolor("#2f6f73")
            cell.set_text_props(color="white", weight="bold")
        else:
            cell.set_facecolor("#f7fbfb" if row % 2 else "white")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(FIG_DIR / "cluster_profile_table.png", dpi=220, bbox_inches="tight", pad_inches=0.08)
    plt.show()


save_response_boxplot(data)
save_profile_table_image(cluster_profiles)


In [ ]:
print("RQ2/RQ3 outputs saved to:", OUT_DIR)
print("\nK-Means diagnostics:")
print(diagnostics.to_string(index=False))
print("\nResponse-time omnibus tests:")
print(omnibus_tests.to_string(index=False))
print("\nResponse-time summary:")
print(response_summary[["cluster_label", "n", "median", "p90"]].to_string(index=False))


## Key Takeaways

- **RQ2:** Manhattan ZIP codes form four interpretable community profiles based on complaint-topic composition.
- **RQ3:** Response times differ significantly across the four profiles (`Kruskal-Wallis p < 0.001`).
- The slower profiles are mainly **Sanitation & Property Conditions** and **Parking & Construction Violations**.
- The faster profiles are mainly **Residential Noise & Street Disruption** and **Building Noise & Vendor Issues**.
